In [8]:
import pandas as pd
import numpy as np

df = pd.read_excel(
    "Данные_для_курсовои_Классическое_МО.xlsx"
)
df = df.drop(columns=["Unnamed: 0"])
df = df.fillna(df.median())

In [9]:
targets = ["IC50, mM", "CC50, mM", "SI"]
X = df.drop(columns=targets)

y = (df["SI"] > df["SI"].median()).astype(int)

In [10]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,random_state=42, stratify=y) #гиперпараметр stratify используем чтобы доля классов сохранялась

In [11]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score)


lr = LogisticRegression(max_iter=5000, C=1.0) #выбрано оптимальное число итераций обучения и оптимальный штраф за слишком сложную модель

lr.fit(X_train, y_train)

pred = lr.predict(X_test)

print("Logistic Regression")

print("Accuracy:", accuracy_score(y_test, pred))

print("F1:", f1_score(y_test, pred))

print("ROC-AUC:", roc_auc_score(y_test, pred))

Logistic Regression
Accuracy: 0.5024875621890548
F1: 0.0
ROC-AUC: 0.5


Модель логистической регресии пока демонстрирует не самые лучшие значения, модель все еще слаба

Проверим модель случайного леса

In [12]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=700, max_depth=10, min_samples_split=5) #глубина оптимальная, нету недообучения и риска переобучения
                                                                                #min_samples_split делает деревья менее сложными борется с переобучением
rf.fit(X_train, y_train)

pred = rf.predict(X_test)

print("Random Forest")

print("Accuracy:", accuracy_score(y_test, pred))

print("F1:", f1_score(y_test, pred))

print("ROC-AUC:", roc_auc_score(y_test, pred))

Random Forest
Accuracy: 0.6517412935323383
F1: 0.6195652173913043
ROC-AUC: 0.6513366336633662


Данные параметры уже демонстрируют что модель вполне хорошо понимает наши признаки и действительно находит закономерности

Посмотрим на градиент бустинг

In [13]:
from sklearn.ensemble import GradientBoostingClassifier

gb = GradientBoostingClassifier(random_state=42, max_depth=10, min_samples_split=5)   #глубина оптимальная, нету недообучения и риска переобучения
                                                                                     #min_samples_split делает деревья менее сложными борется с переобучением
gb.fit(X_train, y_train)

pred = gb.predict(X_test)

print("GradientBoostingClassifier")

print("Accuracy:", accuracy_score(y_test, pred))

print("F1:", f1_score(y_test, pred))

print("ROC-AUC:", roc_auc_score(y_test, pred))

GradientBoostingClassifier
Accuracy: 0.6268656716417911
F1: 0.6031746031746031
ROC-AUC: 0.6265841584158415


Лучшие метрики продемонстрировал случайный лес

Посмотрим как она сработает на 9 наших избранных признаках

In [15]:
selected_features = [
    "VSA_EState8",
    "VSA_EState6",
    "VSA_EState4",
    "MolLogP",
    "qed",
    "FractionCSP3",
    "NumSaturatedHeterocycles",
    "NumSaturatedCarbocycles",
    "NumAliphaticCarbocycles"
]

X_small = df[selected_features]

X_train, X_test, y_train, y_test = train_test_split(
    X_small,
    y,
    test_size=0.2,
    random_state=42
)

In [16]:
rf = RandomForestClassifier(n_estimators=700, max_depth=10, min_samples_split=5)


rf.fit(X_train, y_train)

pred = rf.predict(X_test)

In [17]:
print("RandomForestClassifier")

print("Accuracy:", accuracy_score(y_test, pred))

print("F1:", f1_score(y_test, pred))

print("ROC-AUC:", roc_auc_score(y_test, pred))

RandomForestClassifier
Accuracy: 0.6865671641791045
F1: 0.6227544910179641
ROC-AUC: 0.6777180406212664


F1 почти не изменился, зато Accuracy и ROC-AUC выросли вполне заметно

В итоге можно сказать что наш EDA оказался полезен прежде всего для задач классификации, что предсказуемо учитывая природу наших данных. Учитывая финальное улучшение наших метрик как и в случае с CC50 можно сделать вывод, что лишние шумовые признаки зачастую мешают сильнее, поэтому после их удаления качество может расти.

Теперь сделаем предсказания модели для SI > 8

In [18]:
y = (df["SI"] > 8).astype(int)

Проверим балансировку классов

In [19]:
print(y.value_counts())
print(y.value_counts(normalize=True))

SI
0    644
1    357
Name: count, dtype: int64
SI
0    0.643357
1    0.356643
Name: proportion, dtype: float64


Классы сбалансированны умеренно

In [20]:
lr = LogisticRegression(max_iter=5000, C=1.0)

lr.fit(X_train, y_train)

pred = lr.predict(X_test)

print("Logistic Regression")

print("Accuracy:", accuracy_score(y_test, pred))

print("F1:", f1_score(y_test, pred))

print("ROC-AUC:", roc_auc_score(y_test, pred))

Logistic Regression
Accuracy: 0.5870646766169154
F1: 0.546448087431694
ROC-AUC: 0.5836320191158901


Модель логистической регрессии показывает относительно средние результаты по метрикам

Проверим также для случайного леса и для градиент бустинга

In [21]:
rf = RandomForestClassifier(n_estimators=700, max_depth=10, min_samples_split=5)
rf.fit(X_train, y_train)

pred = rf.predict(X_test)

print("Random Forest")

print("Accuracy:", accuracy_score(y_test, pred))

print("F1:", f1_score(y_test, pred))

print("ROC-AUC:", roc_auc_score(y_test, pred))

Random Forest
Accuracy: 0.6716417910447762
F1: 0.611764705882353
ROC-AUC: 0.6638291517323774


In [22]:
gb = GradientBoostingClassifier(random_state=42, max_depth=10, min_samples_split=5)
gb.fit(X_train, y_train)

pred = gb.predict(X_test)

print("GradientBoostingClassifier")

print("Accuracy:", accuracy_score(y_test, pred))

print("F1:", f1_score(y_test, pred))

print("ROC-AUC:", roc_auc_score(y_test, pred))

GradientBoostingClassifier
Accuracy: 0.6119402985074627
F1: 0.5894736842105263
ROC-AUC: 0.6112604540023894


Лучшие метрики снова демонстрирует случайный лес. Проверяем на наших признаках

In [26]:
selected_features = [
    "VSA_EState8",
    "VSA_EState6",
    "VSA_EState4",
    "MolLogP",
    "qed",
    "FractionCSP3",
    "NumSaturatedHeterocycles",
    "NumSaturatedCarbocycles",
    "NumAliphaticCarbocycles"
]

X_small = df[selected_features]

X_train, X_test, y_train, y_test = train_test_split(
    X_small,
    y,
    test_size=0.2,
    random_state=42
)

In [27]:
rf = RandomForestClassifier(n_estimators=700, max_depth=10, min_samples_split=5)
rf.fit(X_train, y_train)

pred = rf.predict(X_test)

In [29]:
print("RandomForestClassifier")

print("Accuracy:", accuracy_score(y_test, pred))

print("F1:", f1_score(y_test, pred))

print("ROC-AUC:", roc_auc_score(y_test, pred))

RandomForestClassifier
Accuracy: 0.7064676616915423
F1: 0.48695652173913045
ROC-AUC: 0.6356446370530877


Для задачи классификации SI > 8 отбор признаков привёл к неоднозначному результату. Значение Accuracy увеличилось с 0.672 до 0.706, однако метрики F1-score и ROC-AUC снизились. Это свидетельствует о том, что модель стала чаще правильно классифицировать объекты в целом, но хуже выявлять объекты положительного класса. Вероятно, для данной задачи часть информации содержится в дескрипторах, которые не вошли в итоговый набор из 9 признаков, потому для классификации SI > 8 предпочтителен полный набор признаков. В целом это логично, учитывая что самые крайние значения SI могут предпочитать признаки несколько другой в химических соединениях, а 9 наших признаков являются в таком случае лишь крепким фундаментом

В целом же можно сказать, что для задач классификации мы нашли подтверждение, что проведённое EDA для наших признаков корректно и что указанные признаки как уже указывалось могут служить твердым фундаментом при поиске эффективного лекарства